# Choosing the number of factors — held-out knockouts

The third criterion. Twenty per cent of the knockouts are held out, stratified
by guide module; the ICA is fitted on the rest for every number of components
from 1 to 30; and each held-out knockout's response is regressed on the fitted
factors to see how much of it they explain. Six repeats.

Unlike the tests in `01` and the in-sample curve in `02`, this asks what the
factors are worth on perturbations the decomposition never saw.

:::{warning}
The split is **not seeded**: `sample()` draws fresh held-out guides on every
call, so the spread shifts from run to run. The shape of the curve is the
result, not the individual numbers.
:::

## Setup

In [ ]:
source("../paths.R")   # SCREEN_DATA, REFERENCE, TEXTFILES; see data/README.md

# NOTE: Main.R, Conf.R, Utilities.R and DAVID_query.R are deliberately not
# sourced, as in 03. ICtest is not loaded: it breaks every ggplot "+" chain.
library(ica)
library(pheatmap)
library(reshape2)   # melt()
library(ggplot2)
library(factoextra)
library(repr)

`%ni%` <- Negate(`%in%`)
lappend <- function(lst, obj) { lst[[length(lst) + 1]] <- obj; lst }

computeR2 <- function(response, prediction) {
    rss <- sum((response - prediction)^2, na.rm = TRUE)       # residual sum of squares
    tss <- sum((response - mean(response, na.rm = TRUE))^2)   # total sum of squares
    1 - rss / tss
}

geneCutOff   <- 4
targetCutOff <- 15
nFactors     <- 15

## Inputs

The same coefficient matrix and module tables the factorisation uses.

In [ ]:
selCoefs <- read.csv(referencePath("SignificantCoefMatrix.csv"), row.names = 1)
selCoefs <- t(selCoefs)
KOGenes <-  colnames(selCoefs)
KOGenes <- sapply(KOGenes, function(x){strsplit(x,"_")[[1]][2]})
colnames(selCoefs) <- KOGenes

#geneModulesN <- data.frame(read.csv("outputs/ME_GeneModules_6_Modules.csv", row.names = 1))
geneModulesN <- data.frame(read.csv(referencePath("ME_GeneModules_leiden_8_Modules.csv")), row.names = 1)
rownames(geneModulesN) <- geneModulesN$GeneName
geneModulesN$GeneName <- NULL
geneModulesN$GeneGroup <- factor(geneModulesN$GeneGroup)

head(geneModulesN)

#guideModulesN <- data.frame(read.csv("outputs/ME_GuideModules_7_Modules.csv", row.names = 1))
#guideModulesN$GuideName <- sapply(guideModulesN$GuideName, function(x){strsplit(as.character(x),"_")[[1]][2]})

guideModulesN <- data.frame(read.csv(referencePath("ME_GuideModules_leiden_6_Modules.csv")),  row.names = 1)
rownames(guideModulesN) <- guideModulesN$GuideName 

head(guideModulesN)

## The held-out evaluation

One call is one random split across all component counts; the loop below repeats
it six times.

In [ ]:
evaluateFactorNo <- function(){
    allTestGuides = list()

    for(elem in unique(guideModulesN$GuideGroup)){
        myTmp = guideModulesN[guideModulesN$GuideGroup == elem,]
        k = sample(myTmp$GuideName, round(nrow(myTmp) *0.2, digits = 0), replace=FALSE)
        allTestGuides = lappend(allTestGuides, k)
    }

    allTestGuides = unique(unlist(allTestGuides))
    allTrainingGuides = guideModulesN$GuideName[guideModulesN$GuideName %ni% allTestGuides]
    selCoefsTest = data.frame(selCoefs[,allTestGuides])
    selCoefsTrain = data.frame(selCoefs[,allTrainingGuides])
    
    
    allExpVars = list()

    for(i in 1:30){
        k <- icaimax(selCoefsTrain, nc = i, center = T)
        mySelFac = data.frame(k$S)
        
        r2ForNfac= c()
        for(j in 1:ncol(selCoefsTest)){

            mySelFac$y = selCoefsTest[,j]
            k = lm("y ~ .", data= mySelFac)
            myPredict = predict(k, newdata = mySelFac)

            myR2 = computeR2(mySelFac$y, myPredict)
            r2ForNfac = c(r2ForNfac, myR2)
        }
        
        allExpVars = lappend(allExpVars, r2ForNfac)
    }
    
    
    allExpVars = data.frame(do.call(cbind, allExpVars))
    rownames(allExpVars) = colnames(selCoefsTest) 
    colnames(allExpVars) = paste0("NFac_",1:ncol(allExpVars))
    allExpVars$genes = rownames(allExpVars)
    allExpVarsMelted = melt(allExpVars)
    
    return(allExpVarsMelted)

}

In [ ]:
hh <- evaluateFactorNo() 

for(i in 1:5){
    vv = evaluateFactorNo()
    
    hh = rbind(hh,  vv)
}

## Explained variance on unseen knockouts

In [ ]:
options(repr.plot.width=12, repr.plot.height=8)

p <- ggplot(hh, aes(x=variable, y=value)) + 
     geom_violin(trim=FALSE)+
     geom_boxplot()+theme_minimal()+ 
     #stat_summary(fun.data = mean, geom = "text", size=6)+
        # stat_compare_means(method="t.test",paired = FALSE,
        #                         comparisons = list(c("Control", "KO")),
        #                         color="red", method.args = list(alternative = "less"),
        #                         label.y = c(30), aes(label = ..p.signif..))+
     #scale_fill_brewer(palette = "Pastel1")+
ylab("Explained variance in unseen perturbation responses (R^2)")+
theme(axis.text = element_text(size=15),
      axis.text.x = element_text(angle=90),
              axis.title =  element_text(size=16))+ylim(0,1) +xlab("Number of ICA factors") 

pdf("figures/HeldOutExplainedVariance.pdf", width = 12, height = 8)
print(p)
dev.off()

p   # show it in the notebook as well as saving it